In [ ]:
import os
import sys
import time

!{sys.executable} --version
if sys.version_info.minor == 8:
    raise RuntimeError('USE JUPYTER KERNEL VENV 3.10/310/DEFAULT INSTEAD')

!cd /workspace/CRYPTO_MACAQUES && pip install .
!cd /home/crypto/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && source venv/bin/activate && !{sys.executable} -m pip install .

from IPython.core.display_functions import clear_output
clear_output()

In [ ]:
%load_ext autoreload
%autoreload

from datetime import timedelta
from dateutil import parser

from SRC.LIBRARIES.new_data_utils import fetch
from SRC.CORE._CONSTANTS import _KIEV_TIMESTAMP
import SRC.LIBRARIES.new_utils as nu


# TODO закомментированный код внизу
# TODO подтянуть зеленый стоп
# TODO трендовый фильтр должен позволять входить на первой свече видимого диапазона
# TODO Proper order
# TODO Сначала вкоммитить новый функционал
# TODO перебирать выводить следующие параметры в статистику автоматизации
# todo doge/xrp 4h 2026 champions


SYM = 'eth'.upper()
SYMBOL = f'{SYM}USDT'
TF = '15' + 'm'.upper()
CAPITAL_PER_TRADE = 1000
COMMISSION_RATE = 0.00075
SLIPPAGE = 0.0002
FORCE_CLOSE_AT_END = False

# ========== КРАСНЫЙ КАНАЛ ==========
ENABLE_RED_CHANNEL = True
RED_MIN_BREAK_BODY_TO_ATR = 0.15
RED_TP_LEVEL = 'yellow'
RED_TP_MODE = 'fixed'
RED_SL_RATIO = 1
RED_REQUIRE_GREEN_TOUCH_AFTER_SL = False
RED_MIN_BARS_OUTSIDE = 1
RED_STOCH_FILTER = True
RED_STOCH_SERIES = 'D'
RED_STOCH_OVERSOLD = 30
RED_STOCH_OVERBOUGHT = 70

# ========== ЗЕЛЁНЫЙ КАНАЛ ==========
ENABLE_GREEN_CHANNEL = True
GREEN_MIN_BREAK_BODY_TO_ATR = 0.50
GREEN_TP_LEVEL = 'green'
GREEN_TP_MODE = 'fixed'
GREEN_REQUIRE_YELLOW_TOUCH_AFTER_SL = True
GREEN_MIN_BARS_OUTSIDE = 1
GREEN_STOCH_FILTER = False
GREEN_STOCH_SERIES = 'K'
GREEN_STOCH_OVERSOLD = 30
GREEN_STOCH_OVERBOUGHT = 70

market_type = 'SPOT'
mrc_buffer = 1000
stoch_buffer = 50
atr_buffer = 200

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple


# ========== ОРИГИНАЛЬНЫЕ ФУНКЦИИ ==========

def get_df_back(df):
    return df.copy()


def get_enter(df_back, df_counter, band, condition_series, min_bars_outside):
    candle_bodies_outside_band_counter = 0
    len_condition_series = len(condition_series)
    candle_bodies_outside_band = np.zeros(len_condition_series)

    for i in range(len_condition_series):
        if condition_series.iloc[i]:
            candle_bodies_outside_band_counter += 1
        else:
            candle_bodies_outside_band_counter = 0

        candle_bodies_outside_band[i] = candle_bodies_outside_band_counter

    count_series = pd.Series(candle_bodies_outside_band, index=condition_series.index)
    previous_candle_bodies_outside_band_count = np.roll(count_series, 1)
    previous_candle_bodies_outside_band_count = previous_candle_bodies_outside_band_count[len(df_counter) - len(df_back):]
    candle_touches_band = ((df_back['low'] <= df_back[band]) & (df_back['high'] >= df_back[band])).values

    return candle_touches_band & (previous_candle_bodies_outside_band_count >= min_bars_outside)


def get_candle_body_part_below_loband(df_counter, loband):
    return (df_counter['open'] < df_counter[loband]) | (df_counter['close'] < df_counter[loband])


def get_candle_body_part_above_upband(df_counter, upband):
    return (df_counter['open'] > df_counter[upband]) | (df_counter['close'] > df_counter[upband])


def get_long_enter(df_back, df_counter, loband, min_bars_outside):
    return pd.Series(
        get_enter(df_back, df_counter, loband, get_candle_body_part_below_loband(df_counter, loband), min_bars_outside),
        index=df_back.index
    )


def get_short_enter(df_back, df_counter, upband, min_bars_outside):
    return pd.Series(
        get_enter(df_back, df_counter, upband, get_candle_body_part_above_upband(df_counter, upband), min_bars_outside),
        index=df_back.index
    )


def select_stoch_values(df_back: pd.DataFrame, stoch_series: str) -> pd.Series:
    return df_back['stoch_k' if stoch_series.upper() == 'K' else 'stoch_d']


def get_db_back_range(df_back):
    return range(0, len(df_back))


def get_current_candle_data(df_back, i):
    candle = df_back.iloc[i]

    return candle['high'], candle['low'], candle['loband1'], candle['upband1'], candle['loband2'], candle['upband2'], candle['meanline']


def find_break_candle_idx(df_counter, break_idx_band, min_bars_outside, find_break_candle_idx_offset, current_i):
    previous_i = current_i + find_break_candle_idx_offset - 1

    # Определяем условие для тела свечи за каналом
    if 'loband' in break_idx_band:
        condition = get_candle_body_part_below_loband(df_counter, break_idx_band)
    else:
        condition = get_candle_body_part_above_upband(df_counter, break_idx_band)

    counter = 0
    # Ищем начало серии
    for j in range(previous_i, -1, -1):
        if condition.iloc[j]:
            counter += 1
        else:
            break

    # Первая свеча в серии
    first_break_idx = previous_i - counter + 1

    # Проверяем, что серия достаточно длинная
    if counter >= min_bars_outside:
        return first_break_idx

    return -1


def get_break_candle_data(df_counter, break_idx_band, min_bars_outside, find_break_candle_idx_offset, i):
    """Возвращает параметры свечи пробоя"""
    break_idx = find_break_candle_idx(df_counter, break_idx_band, min_bars_outside, find_break_candle_idx_offset, i)

    if break_idx < 0:
        return None

    candle = df_counter.iloc[break_idx]
    body_size = abs(candle['close'] - candle['open'])
    atr_value = candle['atr']

    return {
        'break_idx': break_idx,
        'body_size': body_size,
        'range_size': candle['high'] - candle['low'],
        'body_to_atr': body_size / atr_value if atr_value > 0 else 0
    }


def get_is_long_signal(long_enter, i):
    return long_enter.iloc[i]


def get_is_short_signal(short_enter, i):
    return short_enter.iloc[i]


def get_is_any_type_signal(is_long_signal, is_short_signal):
    return is_long_signal or is_short_signal


def get_price_slipped_up(price, slippage):
    return price * (1 + slippage)


def get_price_slipped_down(price, slippage):
    return price * (1 - slippage)


def get_can_enter_by_break_candle(can_enter, filtered_by_body, break_candle_data, min_break_body_to_atr):
    if break_candle_data and break_candle_data['body_to_atr'] < min_break_body_to_atr:
        can_enter = False
        filtered_by_body += 1

    return can_enter, filtered_by_body


def get_can_enter_by_stoch_filter(can_enter, filtered_by_stoch, stoch_filter, direction, stoch_values, stoch_oversold, stoch_overbought, i):
    if can_enter and stoch_filter:
        current_stoch = stoch_values.iloc[i]

        if (direction == 'long' and not (current_stoch <= stoch_oversold)) or (direction == 'short' and not (current_stoch >= stoch_overbought)):
            can_enter = False
            filtered_by_stoch += 1

    return can_enter, filtered_by_stoch


def get_last_trade_exit_data(trades):
    last_trade = trades[-1]

    return last_trade['exit_idx'], last_trade['exit_type']


def get_candle_touches_price(low, price, high):
    return low <= price <= high


def get_enter_amount(commission, capital_per_trade, entry_price):
    return (capital_per_trade * (1 - commission)) / entry_price


def get_position(direction, channel, i, entry_price, amount, tp_price, sl_price, tp_line, tp_mode, break_candle_data):
    pos = {
        'type': direction,
        'channel': channel,
        'entry_idx': i,
        'entry_price': entry_price,
        'amount': amount,
        'tp': tp_price,
        'sl': sl_price,
        'tp_line': tp_line,
        'tp_mode': tp_mode,
        'tp_distance': abs(entry_price - tp_price)
    }

    if break_candle_data:
        pos['break_body_to_atr'] = break_candle_data['body_to_atr']
        pos['break_idx'] = break_candle_data['break_idx']
    else:
        pos['break_body_to_atr'] = 0
        pos['break_idx'] = -1

    return pos


def calculate_pnl(position: Dict, exit_price: float, slippage: float, commission: float, capital_per_trade: float) -> Tuple[float, float]:
    direction = position['type']
    amount = position['amount']

    if direction == 'long':
        exit_price_slipped = get_price_slipped_down(exit_price, slippage)
        exit_proceeds = amount * exit_price_slipped * (1 - commission)
        return exit_proceeds - capital_per_trade, exit_price_slipped
    elif direction == 'short':
        exit_price_slipped = get_price_slipped_up(exit_price, slippage)
        entry_proceeds = amount * position['entry_price']
        exit_cost = amount * exit_price_slipped * (1 + commission)
        return entry_proceeds - exit_cost, exit_price_slipped


def get_trade(position, exit_idx, exit_price_slipped, pnl, exit_type):
    return {
        'channel': position['channel'],
        'type': position['type'],
        'entry_idx': position['entry_idx'],
        'exit_idx': exit_idx,
        'entry_price': position['entry_price'],
        'exit_price': exit_price_slipped,
        'pnl': pnl,
        'exit_type': exit_type,
        'break_body_to_atr': position.get('break_body_to_atr', 0)
    }


def exit_from_position(position, current_upband1, current_meanline, current_loband1, current_high, current_low, slippage, commission, capital_per_trade, trades, i):
    if position is not None:
        if position['tp_mode'] == 'dynamic':
            if position['tp_line'] == 'upband1':
                position['tp'] = current_upband1
            elif position['tp_line'] == 'meanline':
                position['tp'] = current_meanline
            elif position['tp_line'] == 'loband1':
                position['tp'] = current_loband1

        exit_price = None
        exit_type = None

        for col in ['tp', 'sl']:
            price = position[col]

            if get_candle_touches_price(current_low, price, current_high):
                exit_price = price
                exit_type = col

        if exit_price is not None:
            pnl, exit_price_slipped = calculate_pnl(position, exit_price, slippage, commission, capital_per_trade)
            trades.append(get_trade(position, i, exit_price_slipped, pnl, exit_type))
            position = None

    return position, trades


def force_close_position(position: Dict, df_back: pd.DataFrame, slippage: float, commission: float, capital_per_trade: float) -> Dict:
    exit_price = df_back.iloc[-1]['close']
    pnl, exit_price_slipped = calculate_pnl(position, exit_price, slippage, commission, capital_per_trade)

    return get_trade(position, len(df_back) - 1, exit_price_slipped, pnl, 'forced')


def get_metrics(total_pnl, win_rate, num_trades, profit_factor):
    return {
        'total_pnl': total_pnl,
        'win_rate': win_rate,
        'num_trades': num_trades,
        'profit_factor': profit_factor
    }


def calculate_metrics(trades: List[Dict]) -> Dict:
    if len(trades) == 0:
        return get_metrics(0, 0, 0, 0)

    trades_df = pd.DataFrame(trades)
    total_pnl = trades_df['pnl'].sum()
    avg_return_per_trade_percent = (total_pnl / (capital_per_trade * len(trades_df))) * 100
    win_rate = (trades_df['pnl'] > 0).mean()
    num_trades = len(trades_df)

    sum_profit = trades_df[trades_df['pnl'] > 0]['pnl'].sum()
    sum_loss = abs(trades_df[trades_df['pnl'] < 0]['pnl'].sum())
    profit_factor = sum_profit / sum_loss if sum_loss > 0 else float('inf')

    return get_metrics(total_pnl, win_rate, num_trades, profit_factor)


def backtest_red_channel(
    df: pd.DataFrame,
    df_counter: pd.DataFrame,
    commission: float = 0.00075,
    capital_per_trade: float = 1000.0,
    slippage: float = 0.0002,
    tp_level: str = 'far_green',
    tp_mode: str = 'fixed',
    sl_ratio: float = 3.0,
    require_green_touch_after_sl: bool = False,
    min_bars_outside: int = 0,
    stoch_filter: bool = False,
    stoch_series: str = 'D',
    stoch_oversold: int = 20,
    stoch_overbought: int = 80,
    force_close_at_end: bool = False,
    min_break_body_to_atr: float = 0.0,
    find_break_candle_idx_offset: int = 0
) -> Tuple[List[Dict], Dict]:
    df_back = get_df_back(df)
    long_enter = get_long_enter(df_back, df_counter, 'loband2', min_bars_outside)
    short_enter = get_short_enter(df_back, df_counter, 'upband2', min_bars_outside)
    stoch_values = select_stoch_values(df_back, stoch_series)
    position = None
    trades = []

    # Статистика сигналов
    total_raw_signals = 0
    total_signals_when_free = 0
    filtered_by_body = 0
    filtered_by_stoch = 0
    signals_taken = 0

    for i in get_db_back_range(df_back):
        current_high, current_low, current_loband1, current_upband1, current_loband2, current_upband2, current_meanline = get_current_candle_data(df_back, i)

        is_long_signal = get_is_long_signal(long_enter, i)
        is_short_signal = get_is_short_signal(short_enter, i)

        if get_is_any_type_signal(is_long_signal, is_short_signal):
            total_raw_signals += 1

        if position is None:
            direction = None
            entry_price = None
            tp_price = None
            tp_line = None

            if is_long_signal:
                direction = 'long'
                entry_price = get_price_slipped_up(current_loband2, slippage)
                break_idx_band = 'loband2'

                if tp_level == 'near_green':
                    tp_price = current_loband1
                    tp_line = 'loband1'
                elif tp_level == 'yellow':
                    tp_price = current_meanline
                    tp_line = 'meanline'
                elif tp_level == 'far_green':
                    tp_price = current_upband1
                    tp_line = 'upband1'

            elif is_short_signal:
                direction = 'short'
                entry_price = get_price_slipped_down(current_upband2, slippage)
                break_idx_band = 'upband2'

                if tp_level == 'near_green':
                    tp_price = current_upband1
                    tp_line = 'upband1'
                elif tp_level == 'yellow':
                    tp_price = current_meanline
                    tp_line = 'meanline'
                elif tp_level == 'far_green':
                    tp_price = current_loband1
                    tp_line = 'loband1'

            if direction is not None:
                can_enter = True
                total_signals_when_free += 1
                break_candle_data = get_break_candle_data(df_counter, break_idx_band, min_bars_outside, find_break_candle_idx_offset, i)
                can_enter, filtered_by_body = get_can_enter_by_break_candle(can_enter, filtered_by_body, break_candle_data, min_break_body_to_atr)
                can_enter, filtered_by_stoch = get_can_enter_by_stoch_filter(can_enter, filtered_by_stoch, stoch_filter, direction, stoch_values, stoch_oversold, stoch_overbought, i)

                if can_enter and require_green_touch_after_sl and len(trades) > 0:
                    last_exit_idx, last_exit_type = get_last_trade_exit_data(trades)

                    if last_exit_type == 'sl':
                        touched_green = False

                        for j in range(last_exit_idx + 1, i + 1):
                            low = df_back.iloc[j]['low']
                            high = df_back.iloc[j]['high']
                            loband1_j = df_back.iloc[j]['loband1']
                            upband1_j = df_back.iloc[j]['upband1']

                            if get_candle_touches_price(low, loband1_j, high) or get_candle_touches_price(low, upband1_j, high):
                                touched_green = True
                                break

                        if not touched_green:
                            can_enter = False

                if can_enter:
                    amount = get_enter_amount(commission, capital_per_trade, entry_price)

                    if direction == 'long':
                        distance_to_tp = tp_price - entry_price
                        sl_price = entry_price - distance_to_tp / sl_ratio
                    elif direction == 'short':
                        distance_to_tp = entry_price - tp_price
                        sl_price = entry_price + distance_to_tp / sl_ratio

                    position = get_position(direction, 'red', i, entry_price, amount, tp_price, sl_price, tp_line, tp_mode, break_candle_data)
                    signals_taken += 1

                    continue

        position, trades = exit_from_position(position, current_upband1, current_meanline, current_loband1, current_high, current_low, slippage, commission, capital_per_trade, trades, i)

    # ========== ПРИНУДИТЕЛЬНОЕ ЗАКРЫТИЕ ==========
    if position is not None and force_close_at_end:
        trades.append(force_close_position(position, df_back, slippage, commission, capital_per_trade))

    return trades


def backtest_green_channel(
    df: pd.DataFrame,
    df_counter: pd.DataFrame,
    commission: float = 0.00075,
    capital_per_trade: float = 1000.0,
    slippage: float = 0.0002,
    tp_level: str = 'green',
    tp_mode: str = 'dynamic',
    require_yellow_touch_after_sl: bool = False,
    min_bars_outside: int = 0,
    stoch_filter: bool = False,
    stoch_series: str = 'D',
    stoch_oversold: int = 20,
    stoch_overbought: int = 80,
    force_close_at_end: bool = False,
    min_break_body_to_atr: float = 0.0,
    find_break_candle_idx_offset: int = 0
) -> Tuple[List[Dict], Dict]:
    df_back = get_df_back(df)
    long_enter = get_long_enter(df_back, df_counter, 'loband1', min_bars_outside)
    short_enter = get_short_enter(df_back, df_counter, 'upband1', min_bars_outside)
    stoch_values = select_stoch_values(df_back, stoch_series)
    position = None
    trades = []

    # Статистика сигналов
    total_raw_signals = 0
    total_signals_when_free = 0
    filtered_by_body = 0
    filtered_by_stoch = 0
    signals_taken = 0

    for i in get_db_back_range(df_back):
        current_high, current_low, current_loband1, current_upband1, current_loband2, current_upband2, current_meanline = get_current_candle_data(df_back, i)

        is_long_signal = get_is_long_signal(long_enter, i)
        is_short_signal = get_is_short_signal(short_enter, i)

        if get_is_any_type_signal(is_long_signal, is_short_signal):
            total_raw_signals += 1

        if position is None:
            direction = None
            entry_price = None
            tp_price = None
            tp_line = None

            if is_long_signal:
                direction = 'long'
                entry_price = get_price_slipped_up(current_loband1, slippage)
                break_idx_band = 'loband1'

                if tp_level == 'green':
                    tp_price = current_upband1
                    tp_line = 'upband1'
                elif tp_level == 'yellow':
                    tp_price = current_meanline
                    tp_line = 'meanline'

            elif is_short_signal:
                direction = 'short'
                entry_price = get_price_slipped_down(current_upband1, slippage)
                break_idx_band = 'upband1'

                if tp_level == 'green':
                    tp_price = current_loband1
                    tp_line = 'loband1'
                elif tp_level == 'yellow':
                    tp_price = current_meanline
                    tp_line = 'meanline'

            if direction is not None:
                can_enter = True
                total_signals_when_free += 1
                break_candle_data = get_break_candle_data(df_counter, break_idx_band, min_bars_outside, find_break_candle_idx_offset, i)
                can_enter, filtered_by_body = get_can_enter_by_break_candle(can_enter, filtered_by_body, break_candle_data, min_break_body_to_atr)
                can_enter, filtered_by_stoch = get_can_enter_by_stoch_filter(can_enter, filtered_by_stoch, stoch_filter, direction, stoch_values, stoch_oversold, stoch_overbought, i)

                if can_enter and require_yellow_touch_after_sl and len(trades) > 0:
                    last_exit_idx, last_exit_type = get_last_trade_exit_data(trades)

                    if last_exit_type == 'sl':
                        touched_yellow = False

                        for j in range(last_exit_idx + 1, i + 1):
                            low = df_back.iloc[j]['low']
                            high = df_back.iloc[j]['high']
                            meanline_j = df_back.iloc[j]['meanline']

                            if get_candle_touches_price(low, meanline_j, high):
                                touched_yellow = True
                                break

                        if not touched_yellow:
                            can_enter = False

                if can_enter:
                    amount = get_enter_amount(commission, capital_per_trade, entry_price)

                    if direction == 'long':
                        sl_price = current_loband2
                    elif direction == 'short':
                        sl_price = current_upband2

                    position = get_position(direction, 'green', i, entry_price, amount, tp_price, sl_price, tp_line, tp_mode, break_candle_data)
                    signals_taken += 1

                    continue

        position, trades = exit_from_position(position, current_upband1, current_meanline, current_loband1, current_high, current_low, slippage, commission, capital_per_trade, trades, i)

    # ========== ПРИНУДИТЕЛЬНОЕ ЗАКРЫТИЕ ==========
    if position is not None and force_close_at_end:
        trades.append(force_close_position(position, df_back, slippage, commission, capital_per_trade))

    return trades

In [ ]:
# ========== ПЕРИОДЫ ДЛЯ ТЕСТИРОВАНИЯ ==========
# periods = [
#     ("2023-1-1 00:00", "2023-6-1 00:00"),
#     ("2023-6-1 00:00", "2024-1-1 00:00"),
#     ("2024-1-1 00:00", "2024-6-1 00:00"),
#     ("2024-6-1 00:00", "2025-1-1 00:00"),
#     ("2025-1-1 00:00", "2025-6-1 00:00"),
#     ("2025-6-1 00:00", "2026-1-1 00:00"),
#     ("2026-1-1 00:00", "2026-6-1 00:00")
# ]
periods = [("2026-1-1 0:0", "2026-6-1 0:0")]

In [ ]:
# ========== ЗАПУСК ТЕСТОВ ДЛЯ ВСЕХ ПЕРИОДОВ ==========
all_results = {}

for start_date_str, end_date_str in periods:
    print(f"\n{'='*80}")
    print(f"ПЕРИОД: {start_date_str} - {end_date_str}")
    
    display_start_dt = parser.parse(start_date_str)
    load_end_date = parser.parse(end_date_str)
    
    tf_value = int(TF[:-1])
    timeframe_seconds = {
        'M': tf_value * 60,
        'H': tf_value * 3600,
        'D': tf_value * 86400
    }.get(TF[-1], 1800)
    
    load_start_dt = display_start_dt - timedelta(seconds=timeframe_seconds * mrc_buffer)
    
    mrc_df = fetch(market_type, SYMBOL, TF, load_start_dt, load_end_date)
    df = mrc_df.iloc[mrc_buffer:].copy()
    
    # Убеждаемся, что индексы уникальны
    mrc_df = mrc_df[~mrc_df.index.duplicated(keep='first')]
    df = df[~df.index.duplicated(keep='first')]
    
    # Stochastic
    stoch_calc_df = nu.prepare_buffer_data(mrc_df, df, stoch_buffer)
    df = nu.stochastic_tradingview(df, stoch_calc_df)
    
    # ATR
    atr_calc_df = nu.prepare_buffer_data(mrc_df, df, atr_buffer)
    df = nu.atr(df, atr_calc_df)
    
    # Расчёт MRC
    mrc_df = nu.mrc_calculate(mrc_df, df)
    
    # Сохраняем df_counter
    df_counter = mrc_df.iloc[mrc_buffer - 1 - max(RED_MIN_BARS_OUTSIDE, GREEN_MIN_BARS_OUTSIDE):].copy()

    # ATR df_counter
    atr_calc_df_counter = nu.prepare_buffer_data(mrc_df, df_counter, atr_buffer)
    df_counter = nu.atr(df_counter, atr_calc_df_counter)

    all_trades = []

    find_break_candle_idx_offset = len(df_counter) - len(df)
        
    if ENABLE_RED_CHANNEL:
        red_trades, _ = backtest_red_channel(
            df=df,
            df_counter=df_counter,
            commission=COMMISSION_RATE,
            capital_per_trade=CAPITAL_PER_TRADE,
            slippage=SLIPPAGE,
            tp_level=RED_TP_LEVEL,
            tp_mode=RED_TP_MODE,
            sl_ratio=RED_SL_RATIO,
            require_green_touch_after_sl=RED_REQUIRE_GREEN_TOUCH_AFTER_SL,
            min_bars_outside=RED_MIN_BARS_OUTSIDE,
            stoch_filter=RED_STOCH_FILTER,
            stoch_series=RED_STOCH_SERIES,
            stoch_oversold=RED_STOCH_OVERSOLD,
            stoch_overbought=RED_STOCH_OVERBOUGHT,
            force_close_at_end=FORCE_CLOSE_AT_END,
            min_break_body_to_atr=RED_MIN_BREAK_BODY_TO_ATR,
            find_break_candle_idx_offset=find_break_candle_idx_offset
        )
        all_trades.extend(red_trades)
        
    if ENABLE_GREEN_CHANNEL:
        green_trades, _ = backtest_green_channel(
            df=df,
            df_counter=df_counter,
            commission=COMMISSION_RATE,
            capital_per_trade=CAPITAL_PER_TRADE,
            slippage=SLIPPAGE,
            tp_level=GREEN_TP_LEVEL,
            tp_mode=GREEN_TP_MODE,
            require_yellow_touch_after_sl=GREEN_REQUIRE_YELLOW_TOUCH_AFTER_SL,
            min_bars_outside=GREEN_MIN_BARS_OUTSIDE,
            stoch_filter=GREEN_STOCH_FILTER,
            stoch_series=GREEN_STOCH_SERIES,
            stoch_oversold=GREEN_STOCH_OVERSOLD,
            stoch_overbought=GREEN_STOCH_OVERBOUGHT,
            force_close_at_end=FORCE_CLOSE_AT_END,
            min_break_body_to_atr=GREEN_MIN_BREAK_BODY_TO_ATR,
            find_break_candle_idx_offset=find_break_candle_idx_offset
        )
        all_trades.extend(green_trades)
        
    metrics = calculate_metrics(all_trades)
    print(f"Total pnL = ${metrics['total_pnl']:.2f}, Trades = {metrics['num_trades']}, WR = {metrics['win_rate']:.2f} PF = {metrics['profit_factor']:.2f}")

print("\n" + "="*100)

In [ ]:
# %load_ext autoreload
# %autoreload
#
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots
#
# import SRC.LIBRARIES.new_indicator_plot_utils as nipu
#
# # ========== НАСТРОЙКИ ОТОБРАЖЕНИЯ ==========
# is_log_scale_by_default = False
# candlestick_row = 1
# volume_row = 2
# stoch_row = 3
# atr_row = 4
#
# # ========== СОЗДАНИЕ ГРАФИКА ==========
# fig_main = make_subplots(
#     rows=4, cols=1,
#     shared_xaxes=True,
#     row_heights=[3, 0.7, 1.5, 1],
#     vertical_spacing=0.03
# )
#
# fig_main.add_trace(
#     go.Candlestick(
#         x=df[_KIEV_TIMESTAMP],
#         open=df["open"],
#         high=df["high"],
#         low=df["low"],
#         close=df["close"],
#         name="OHLC"
#     ),
#     row=candlestick_row, col=1
# )
#
# nipu.add_mrc(candlestick_row, fig_main, df)
# nipu.add_volume(df, volume_row, fig_main)
# nipu.add_stoch(stoch_row, fig_main, df, RED_STOCH_OVERBOUGHT, RED_STOCH_OVERSOLD)
# nipu.add_atr(atr_row, fig_main, df)
#
# # ========== ЗАПУСК БЭКТЕСТА ==========
# all_trades = []
#
# if ENABLE_RED_CHANNEL:
#     red_trades = backtest_red_channel(
#         df=df,
#         df_counter=df_counter,
#         commission=COMMISSION_RATE,
#         capital_per_trade=CAPITAL_PER_TRADE,
#         slippage=SLIPPAGE,
#         tp_level=RED_TP_LEVEL,
#         tp_mode=RED_TP_MODE,
#         sl_ratio=RED_SL_RATIO,
#         require_green_touch_after_sl=RED_REQUIRE_GREEN_TOUCH_AFTER_SL,
#         min_bars_outside=RED_MIN_BARS_OUTSIDE,
#         stoch_filter=RED_STOCH_FILTER,
#         stoch_series=RED_STOCH_SERIES,
#         stoch_oversold=RED_STOCH_OVERSOLD,
#         stoch_overbought=RED_STOCH_OVERBOUGHT,
#         force_close_at_end=FORCE_CLOSE_AT_END
#     )
#     all_trades.extend(red_trades)
# print(f"Красный канал: {len(red_trades)} сделок")
#
# if ENABLE_GREEN_CHANNEL:
#     green_trades = backtest_green_channel(
#         df=df,
#         df_counter=df_counter,
#         commission=COMMISSION_RATE,
#         capital_per_trade=CAPITAL_PER_TRADE,
#         slippage=SLIPPAGE,
#         tp_level=GREEN_TP_LEVEL,
#         tp_mode=GREEN_TP_MODE,
#         require_yellow_touch_after_sl=GREEN_REQUIRE_YELLOW_TOUCH_AFTER_SL,
#         min_bars_outside=GREEN_MIN_BARS_OUTSIDE,
#         stoch_filter=GREEN_STOCH_FILTER,
#         stoch_series=GREEN_STOCH_SERIES,
#         stoch_oversold=GREEN_STOCH_OVERSOLD,
#         stoch_overbought=GREEN_STOCH_OVERBOUGHT,
#         force_close_at_end=FORCE_CLOSE_AT_END
#     )
#     all_trades.extend(green_trades)
#     print(f"Зелёный канал: {len(green_trades)} сделок")
#
# # Общая статистика
# metrics = calculate_metrics(all_trades, CAPITAL_PER_TRADE)
#
# # ========== ВЫВОД РЕЗУЛЬТАТОВ ==========
# print(f"\nРезультаты красного канала (обратное пересечение красной линии):")
# print(f"{SYM} {TF} | {DISPLAY_START_DATE_STR} - {DISPLAY_END_DATE_STR}")
# print(f"RED_TP_LEVEL: {RED_TP_LEVEL}")
# print(f"RED_TP_MODE: {RED_TP_MODE}")
# print(f"RED_SL_RATIO: 1:{RED_SL_RATIO}")
# print(f"Требовать касание зелёной после выхода: {RED_REQUIRE_GREEN_TOUCH_AFTER_SL}")
# print(f"RED_STOCH_FILTER: {RED_STOCH_FILTER} (RED_STOCH_SERIES={RED_STOCH_SERIES}, RED_STOCH_OVERSOLD={RED_STOCH_OVERSOLD}, RED_STOCH_OVERBOUGHT={RED_STOCH_OVERBOUGHT})")
#
# if ENABLE_GREEN_CHANNEL:
#     print(f"\nЗелёный канал: TP={GREEN_TP_LEVEL}({GREEN_TP_MODE}), мин.свечей за зелёной={GREEN_MIN_BARS_OUTSIDE}")
#
# print("-" * 40)
#
# for k, v in metrics.items():
#     if isinstance(v, float):
#         print(f"{k}: {v:.4f}")
#     else:
#         print(f"{k}: {v}")
#
# # Анализ по каналам
# if ENABLE_RED_CHANNEL:
#     if len(red_trades) > 0:
#         red_df = pd.DataFrame(red_trades)
#         red_pnl = red_df['pnl'].sum()
#         red_win_rate = (red_df['pnl'] > 0).mean()
#         print(f"\n🔴 КРАСНЫЙ КАНАЛ:")
#         print(f"  Сделок: {len(red_trades)}")
#         print(f"  PnL: {red_pnl:.2f}$")
#         print(f"  Win Rate: {red_win_rate:.1%}")
#         print(f"  Avg Win: {red_df[red_df['pnl'] > 0]['pnl'].mean():.2f}$")
#         print(f"  Avg Loss: {red_df[red_df['pnl'] < 0]['pnl'].mean():.2f}$")
#         print(f"  Profit Factor: {abs(red_df[red_df['pnl'] > 0]['pnl'].sum() / red_df[red_df['pnl'] < 0]['pnl'].sum()) if red_df[red_df['pnl'] < 0]['pnl'].sum() != 0 else 0:.2f}")
#
# if ENABLE_GREEN_CHANNEL:
#     if len(green_trades) > 0:
#         green_df = pd.DataFrame(green_trades)
#         green_pnl = green_df['pnl'].sum()
#         green_win_rate = (green_df['pnl'] > 0).mean()
#         print(f"\n🟢 ЗЕЛЁНЫЙ КАНАЛ:")
#         print(f"  Сделок: {len(green_trades)}")
#         print(f"  PnL: {green_pnl:.2f}$")
#         print(f"  Win Rate: {green_win_rate:.1%}")
#         print(f"  Avg Win: {green_df[green_df['pnl'] > 0]['pnl'].mean():.2f}$")
#         print(f"  Avg Loss: {green_df[green_df['pnl'] < 0]['pnl'].mean():.2f}$")
#         print(f"  Profit Factor: {abs(green_df[green_df['pnl'] > 0]['pnl'].sum() / green_df[green_df['pnl'] < 0]['pnl'].sum()) if green_df[green_df['pnl'] < 0]['pnl'].sum() != 0 else 0:.2f}")
#
# if ENABLE_RED_CHANNEL and ENABLE_GREEN_CHANNEL:
#     if len(red_trades) > 0 and len(green_trades) > 0:
#         print(f"\n📊 СРАВНЕНИЕ:")
#         print(f"  Разница в PnL: {abs(red_pnl - green_pnl):.2f}$")
#         print(f"  Лучший канал: {'🔴 RED' if red_pnl > green_pnl else '🟢 GREEN'}")
#
# # ========== НАСТРОЙКИ ГРАФИКА ==========
# price_log_scale_value = "log"
# price_linear_scale_value = "linear"
# price_log_scale_title = "Price (log scale)"
# price_linear_scale_title = "Price (linear scale)"
#
# fig_main.update_layout(
#     title=f"🐵 {TF} | TP={RED_TP_LEVEL}({RED_TP_MODE}) | SL=1:{RED_SL_RATIO}",
#     xaxis_rangeslider_visible=False,
#     yaxis1_type=price_log_scale_value if is_log_scale_by_default else price_linear_scale_value,
#     yaxis1_title=price_log_scale_title if is_log_scale_by_default else price_linear_scale_title,
#     yaxis2_title="Volume",
#     yaxis3_title="Stoch (Red channel)",
#     yaxis4_title="ATR",
#     height=1200,
#     bargap=0,
#     updatemenus=[
#         dict(
#             type="buttons",
#             direction="right",
#             active=1 if is_log_scale_by_default else 0,
#             x=0.315,
#             y=1.073,
#             buttons=[
#                 dict(
#                     label="Linear",
#                     method="relayout",
#                     args=[{"yaxis.type": price_linear_scale_value, "yaxis.title.text": price_linear_scale_title}]
#                 ),
#                 dict(
#                     label="Log",
#                     method="relayout",
#                     args=[{"yaxis.type": price_log_scale_value, "yaxis.title.text": price_log_scale_title}]
#                 )
#             ],
#             font=dict(color="red", size=12, family="Arial"),
#             bgcolor="rgba(0, 0, 0, 0)",
#             bordercolor="red",
#             borderwidth=1
#         )
#     ]
# )
#
# # ========== МАРКЕРЫ СДЕЛОК ==========
#
# # Собираем данные для маркеров
# trades_df = pd.DataFrame(all_trades) if all_trades else pd.DataFrame()
#
# def go_scatter(type_col, type_name, group_idx, group_price, name, symbol, size, color, opacity):
#     for channel, group in trades_df[trades_df[type_col] == type_name].groupby('channel'):
#         fig_main.add_trace(
#             go.Scatter(
#                 x=df[_KIEV_TIMESTAMP].iloc[group[group_idx]],
#                 y=group[group_price],
#                 mode='markers',
#                 name=f'{name} ({channel})',
#                 marker=dict(
#                     symbol=symbol,
#                     size=size,
#                     color=color,
#                     opacity=opacity,
#                     line=dict(width=2, color='darkred' if channel == 'red' else 'darkgreen')
#                 )
#             ),
#             row=candlestick_row, col=1
#         )
#
# if len(trades_df) > 0:
#     go_scatter('type', 'long', 'entry_idx', 'entry_price', 'Long Entry', 'triangle-up', 10, 'green', 1)
#     go_scatter('type', 'short', 'entry_idx', 'entry_price', 'Short Entry', 'triangle-down', 10, 'red', 1)
#     go_scatter('exit_type', 'tp', 'exit_idx', 'exit_price', 'Take Profit', 'circle', 10, 'lime', 0.7)
#     go_scatter('exit_type', 'sl', 'exit_idx', 'exit_price', 'Stop Loss', 'x', 12, 'red', 0.7)
#
# fig_main.show()
#
# os.system('afplay /System/Library/Sounds/Glass.aiff')